# DPA - Volume (Part 1): Network Traffic Analysis

Our research questions:

1. What traffic patterns distinguish normal activity from attacks?
2. Which network services and traffic characteristics are associated with different attack categories?

We have six steps so far:

1. Check the source files and small samples.
2. Define the column names and types.
3. Test reading CSV and saving Parquet with a small sample.
4. Read all the data, clean it, and save the row counts.
5. Add IANA service names to the cleaned traffic.
6. Compare traffic patterns and services using Spark.

Run the cells from top to bottom. Keep `DOWNLOAD_DATA = False` if you already have the data. Step 1 uses Python only. The next steps need PySpark, and Step 3 also needs Java. Our Docker setup includes both.


## Optional - Download the data

Part 1 uses the original traffic files in `data/Dataset Network Traffic/` and the IANA registry. The traffic files are about 6.89 GB.

Set `DOWNLOAD_DATA = True` to download them. Existing nonempty files are skipped. If a download stops, rerun this cell; that file starts again. Downloading needs Python only.


In [1]:
DOWNLOAD_DATA = False  # Set to True if you need to download the data.

from pathlib import Path

project_folder = Path.cwd().resolve()
if project_folder.name == "notebooks":
    project_folder = project_folder.parent
elif (project_folder / "DPA-Volume-Part1-Project" / "downloaders").is_dir():
    project_folder = project_folder / "DPA-Volume-Part1-Project"
if not (project_folder / "downloaders" / "download_data.py").is_file():
    raise FileNotFoundError("Open the notebook from the project folder or notebooks/.")

data_folder = project_folder / "data"
parquet_folder = data_folder / "parquet"
print("Project folder:", project_folder)

if DOWNLOAD_DATA:
    import runpy
    downloader = runpy.run_path(str(project_folder / "downloaders" / "download_data.py"))
    downloader["download_traffic"](data_folder)
    downloader["download_iana"](data_folder)
    print("Data is ready.")
else:
    print("Download skipped.")


Project folder: /home/jovyan/work
Download skipped.


## Step 1 - Check the source files

We check the columns and some values before choosing the types and cleaning rules.

For traffic, we read up to 100 rows at five positions in each file: the start, 25%, 50%, 75%, and 99%. These small samples help us check the data. They are not random samples, so we cannot use them to estimate the number of attacks in the full dataset.

IANA is small, so we check the whole file. Nothing is cleaned in this step.


In [2]:
import csv
import io
import json
from collections import Counter
from datetime import datetime
from itertools import islice

traffic_folder = data_folder / "Dataset Network Traffic"
traffic_files = sorted(traffic_folder.glob("*.csv"))
iana_file = data_folder / "Dataset Service and Port" / "service-names-port-numbers.csv"
assert traffic_files, f"No traffic CSVs found in {traffic_folder}"
assert iana_file.is_file(), f"IANA file not found: {iana_file}"

print("Traffic CSVs:", len(traffic_files))
for path in traffic_files:
    print(f"  {path.name}: {path.stat().st_size / 1024**3:.2f} GiB")
print("IANA:", iana_file.name)

Traffic CSVs: 10
  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 0.33 GiB
  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 0.31 GiB
  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 0.36 GiB
  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 3.78 GiB
  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 0.10 GiB
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 0.35 GiB
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 0.36 GiB
  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 0.33 GiB
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 0.31 GiB
  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 0.19 GiB
IANA: service-names-port-numbers.csv


### 1A - Check the headers

We check column names, their order, and repeated names. Nine files have 80 columns. The 20 February file has 84, with four extra columns: `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`. We will define a schema for each layout.


In [3]:
traffic_headers = {}
header_summary = []
for path in traffic_files:
    with path.open(encoding="utf-8-sig", newline="") as source:
        header = next(csv.reader(source))
    assert len(header) == len(set(header)), f"Duplicate column names in {path.name}"
    traffic_headers[path.name] = header
    reference_header = traffic_headers[traffic_files[0].name]
    header_summary.append({
        "file": path.name,
        "size_bytes": path.stat().st_size,
        "column_count": len(header),
        "matches_first_header": header == reference_header,
    })
    print(f"{path.name}: {len(header)} columns; same names/order = {header == reference_header}")

print("\nColumns in the first traffic file:")
for position, name in enumerate(reference_header, start=1):
    print(f"{position:2}. {name}")

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 84 columns; same names/order = False
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True

Columns in the first traffic file:
 1. Dst Port
 2. Protocol
 3. Timestamp
 4. Flow Duration
 5. Tot Fwd Pkts
 6. Tot Bwd Pkts
 7. 

### 1B - Read small samples

`seek` lets us jump to different parts of a large file. We skip the partial line at that position, then read up to 100 rows. We check that each row has the right number of columns and keep the samples grouped by file.


In [4]:
sample_size = 100
sample_positions = (0.0, 0.25, 0.5, 0.75, 0.99)

def read_sample(path, header, fraction):
    with path.open("rb") as source:
        source.seek(int(path.stat().st_size * fraction))
        source.readline()  # Skip the header, or the partial line at this position.
        with io.TextIOWrapper(source, encoding="utf-8", newline="") as text:
            records = list(islice(csv.reader(text), sample_size))
    assert all(len(row) == len(header) for row in records), f"Unexpected CSV record width in {path.name}"
    return [dict(zip(header, row)) for row in records]

sample_rows = []
file_samples = {}
window_summary = []
for path in traffic_files:
    file_samples[path.name] = []
    for fraction in sample_positions:
        rows = read_sample(path, traffic_headers[path.name], fraction)
        sample_rows.extend(rows)
        file_samples[path.name].extend(rows)
        window_summary.append({"file": path.name, "position_fraction": fraction, "sampled_rows": len(rows)})
    print(f"{path.name}: inspected {len(file_samples[path.name])} records")
print("Total inspected traffic records:", len(sample_rows))

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Total inspected traffic records: 5000


### 1C - Check labels, protocols, dates, and numbers

These counts are for the samples only. We keep the original attack labels. Traffic uses numeric protocol codes, which we will match to IANA's protocol names later.

Dates use day/month/year, sometimes with fractional seconds. We keep them as text for now. A negative TCP window value such as `-1` may be a special value, so we should not treat every negative number as an error.


In [5]:
sample_labels = Counter(row["Label"] for row in sample_rows)
sample_protocols = Counter(row["Protocol"] for row in sample_rows)
print("Labels in inspected windows:", dict(sample_labels))
print("Raw protocol codes:", dict(sample_protocols))

formats = ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M:%S.%f")
format_counts = Counter()
parsed_times = []
unparsed_times = []
for row in sample_rows:
    for timestamp_format in formats:
        try:
            parsed_times.append(datetime.strptime(row["Timestamp"], timestamp_format))
            format_counts[timestamp_format] += 1
            break
        except ValueError:
            continue
    else:
        unparsed_times.append(row["Timestamp"])
print("Timestamp formats matched:", dict(format_counts))
print("Unparsed sampled timestamps:", len(unparsed_times))
if parsed_times:
    print("Sample timestamp range:", min(parsed_times), "to", max(parsed_times))

example_columns = ["Dst Port", "Protocol", "Timestamp", "Flow Duration",
                   "Tot Fwd Pkts", "Tot Bwd Pkts", "Label"]
examples = {}
for row in sample_rows:
    if row["Label"] not in examples:
        examples[row["Label"]] = {column: row[column] for column in example_columns}
print("\nOne raw example per observed label:")
print(json.dumps(examples, indent=2))

sentinel_examples = {}
for column in ("Init Fwd Win Byts", "Init Bwd Win Byts"):
    sentinel_examples[column] = sorted({row[column] for row in sample_rows if row[column].startswith("-")})
print("Observed negative values requiring interpretation:", sentinel_examples)


Labels in inspected windows: {'Benign': 4140, 'DoS attacks-SlowHTTPTest': 103, 'DoS attacks-Hulk': 164, 'DDoS attacks-LOIC-HTTP': 37, 'Infilteration': 144, 'DoS attacks-GoldenEye': 56, 'FTP-BruteForce': 6, 'SSH-Bruteforce': 99, 'DDOS attack-HOIC': 251}
Raw protocol codes: {'6': 3511, '0': 325, '17': 1164}
Timestamp formats matched: {'%d/%m/%Y %H:%M:%S': 5000}
Unparsed sampled timestamps: 0
Sample timestamp range: 2018-02-14 01:22:05 to 2018-03-02 12:57:21

One raw example per observed label:
{
  "Benign": {
    "Dst Port": "443",
    "Protocol": "6",
    "Timestamp": "02/03/2018 08:47:38",
    "Flow Duration": "141385",
    "Tot Fwd Pkts": "9",
    "Tot Bwd Pkts": "7",
    "Label": "Benign"
  },
  "DoS attacks-SlowHTTPTest": {
    "Dst Port": "21",
    "Protocol": "6",
    "Timestamp": "16/02/2018 10:12:14",
    "Flow Duration": "21",
    "Tot Fwd Pkts": "1",
    "Tot Bwd Pkts": "1",
    "Label": "DoS attacks-SlowHTTPTest"
  },
  "DoS attacks-Hulk": {
    "Dst Port": "80",
    "Protoco

### 1D - Check IANA

We check missing service names, ports, and protocols. Some ports are ranges, and several service names can share the same port and protocol. We only count these cases here.

IANA tells us which services are registered for a port. It does not prove which application produced the traffic.


In [6]:
with iana_file.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    iana_header = reader.fieldnames
    iana_records = list(reader)  # Only the small IANA registry, never the traffic files.
print("IANA columns:", iana_header)
assert len(iana_header) == len(set(iana_header)), "Duplicate IANA column names"
assert {"Port Number", "Transport Protocol", "Service Name"}.issubset(iana_header)
assert all(None not in row and None not in row.values() for row in iana_records), "Unexpected IANA record width"

port_counts = Counter((row["Port Number"], row["Transport Protocol"]) for row in iana_records)
range_ports = [row["Port Number"] for row in iana_records if "-" in row["Port Number"]]
iana_summary = {
    "file": iana_file.name,
    "row_count": len(iana_records),
    "column_count": len(iana_header),
    "transport_protocol_counts": dict(Counter(row["Transport Protocol"] for row in iana_records)),
    "rows_with_port_ranges": len(range_ports),
    "port_range_examples": list(dict.fromkeys(range_ports))[:5],
    "rows_with_blank_port": sum(not row["Port Number"].strip() for row in iana_records),
    "rows_with_blank_protocol": sum(not row["Transport Protocol"].strip() for row in iana_records),
    "rows_with_blank_service": sum(not row["Service Name"].strip() for row in iana_records),
    "repeated_literal_key_count": sum(count > 1 for count in port_counts.values()),
    "extra_rows_on_repeated_literal_keys": sum(count - 1 for count in port_counts.values()),
}
print(json.dumps(iana_summary, indent=2))
print("Example registered service:", next(row for row in iana_records if row["Service Name"] and row["Port Number"].isdigit()))

IANA columns: ['Service Name', 'Port Number', 'Transport Protocol', 'Description', 'Assignee', 'Contact', 'Registration Date', 'Modification Date', 'Reference', 'Service Code', 'Unauthorized Use Reported', 'Assignment Notes']
{
  "file": "service-names-port-numbers.csv",
  "row_count": 14536,
  "column_count": 12,
  "transport_protocol_counts": {
    "tcp": 6608,
    "udp": 6357,
    "sctp": 93,
    "dccp": 11,
    "": 1467
  },
  "rows_with_port_ranges": 766,
  "port_range_examples": [
    "225-241",
    "249-255",
    "272-279",
    "288-299",
    "301-307"
  ],
  "rows_with_blank_port": 1073,
  "rows_with_blank_protocol": 1467,
  "rows_with_blank_service": 1724,
  "repeated_literal_key_count": 230,
  "extra_rows_on_repeated_literal_keys": 1300
}
Example registered service: {'Service Name': 'tcpmux', 'Port Number': '1', 'Transport Protocol': 'tcp', 'Description': 'TCP Port Service Multiplexer', 'Assignee': '[Mark_Lottor]', 'Contact': '[Mark_Lottor]', 'Registration Date': '', 'Modific

### 1E - Save what we found

Save the inspection summary as JSON and the header checks as CSV. These files describe the inspection; they are not the final analysis or cleaning results.


In [7]:
inspection_folder = project_folder / "reports" / "inspection" / "original"
inspection_folder.mkdir(parents=True, exist_ok=True)
inspection_summary = {
    "scope": "Source inspection only; no cleaning, typed ingestion, or Spark analysis performed",
    "traffic": {
        "source_folder": str(traffic_folder.relative_to(project_folder)),
        "file_count": len(traffic_files),
        "total_file_bytes": sum(path.stat().st_size for path in traffic_files),
        "header_summary": header_summary,
        "headers_by_file": traffic_headers,
        "sampling_method": "100 records after each byte position: 0%, 25%, 50%, 75%, 99%; not statistically representative",
        "window_summary": window_summary,
        "sample_row_count": len(sample_rows),
        "sample_label_counts": dict(sample_labels),
        "sample_protocol_counts": dict(sample_protocols),
        "sample_timestamp_formats": dict(format_counts),
        "sample_timestamp_min": min(parsed_times).isoformat() if parsed_times else None,
        "sample_timestamp_max": max(parsed_times).isoformat() if parsed_times else None,
        "sample_unparsed_timestamp_count": len(unparsed_times),
        "sample_negative_sentinel_values": sentinel_examples,
        "raw_examples_by_observed_label": examples,
    },
    "iana": iana_summary,
}
summary_file = inspection_folder / "inspection_summary.json"
summary_file.write_text(json.dumps(inspection_summary, indent=2), encoding="utf-8")
with (inspection_folder / "traffic_headers.csv").open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(header_summary[0]))
    writer.writeheader()
    writer.writerows(header_summary)
print("Saved:", summary_file)
print("Saved:", inspection_folder / "traffic_headers.csv")

Saved: /home/jovyan/work/reports/inspection/original/inspection_summary.json
Saved: /home/jovyan/work/reports/inspection/original/traffic_headers.csv


Step 1 is complete. Next, we define the schemas.


## Step 2 - Define the schemas

A schema tells Spark the name and type of each column. We write the types ourselves:

- `StringType`: labels, dates before cleaning, and IP addresses.
- `IntegerType`: ports and protocol codes.
- `LongType`: whole-number counts, flags, header lengths, and flow durations.
- `DoubleType`: rates, averages, ratios, and values with decimals or scientific notation.

For example, some byte totals are written as `0.0`, so they need `DoubleType`. `True` allows missing values, which we will check during cleaning. Dates stay as text until then.

We define one schema with 80 columns and another with the four extra columns.


In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType

traffic_schema = StructType([
    StructField("Dst Port", IntegerType(), True),
    StructField("Protocol", IntegerType(), True),
    StructField("Timestamp", StringType(), True),
    StructField("Flow Duration", LongType(), True),
    StructField("Tot Fwd Pkts", LongType(), True),
    StructField("Tot Bwd Pkts", LongType(), True),
    StructField("TotLen Fwd Pkts", DoubleType(), True),
    StructField("TotLen Bwd Pkts", DoubleType(), True),
    StructField("Fwd Pkt Len Max", DoubleType(), True),
    StructField("Fwd Pkt Len Min", DoubleType(), True),
    StructField("Fwd Pkt Len Mean", DoubleType(), True),
    StructField("Fwd Pkt Len Std", DoubleType(), True),
    StructField("Bwd Pkt Len Max", DoubleType(), True),
    StructField("Bwd Pkt Len Min", DoubleType(), True),
    StructField("Bwd Pkt Len Mean", DoubleType(), True),
    StructField("Bwd Pkt Len Std", DoubleType(), True),
    StructField("Flow Byts/s", DoubleType(), True),
    StructField("Flow Pkts/s", DoubleType(), True),
    StructField("Flow IAT Mean", DoubleType(), True),
    StructField("Flow IAT Std", DoubleType(), True),
    StructField("Flow IAT Max", DoubleType(), True),
    StructField("Flow IAT Min", DoubleType(), True),
    StructField("Fwd IAT Tot", DoubleType(), True),
    StructField("Fwd IAT Mean", DoubleType(), True),
    StructField("Fwd IAT Std", DoubleType(), True),
    StructField("Fwd IAT Max", DoubleType(), True),
    StructField("Fwd IAT Min", DoubleType(), True),
    StructField("Bwd IAT Tot", DoubleType(), True),
    StructField("Bwd IAT Mean", DoubleType(), True),
    StructField("Bwd IAT Std", DoubleType(), True),
    StructField("Bwd IAT Max", DoubleType(), True),
    StructField("Bwd IAT Min", DoubleType(), True),
    StructField("Fwd PSH Flags", LongType(), True),
    StructField("Bwd PSH Flags", LongType(), True),
    StructField("Fwd URG Flags", LongType(), True),
    StructField("Bwd URG Flags", LongType(), True),
    StructField("Fwd Header Len", LongType(), True),
    StructField("Bwd Header Len", LongType(), True),
    StructField("Fwd Pkts/s", DoubleType(), True),
    StructField("Bwd Pkts/s", DoubleType(), True),
    StructField("Pkt Len Min", DoubleType(), True),
    StructField("Pkt Len Max", DoubleType(), True),
    StructField("Pkt Len Mean", DoubleType(), True),
    StructField("Pkt Len Std", DoubleType(), True),
    StructField("Pkt Len Var", DoubleType(), True),
    StructField("FIN Flag Cnt", LongType(), True),
    StructField("SYN Flag Cnt", LongType(), True),
    StructField("RST Flag Cnt", LongType(), True),
    StructField("PSH Flag Cnt", LongType(), True),
    StructField("ACK Flag Cnt", LongType(), True),
    StructField("URG Flag Cnt", LongType(), True),
    StructField("CWE Flag Count", LongType(), True),
    StructField("ECE Flag Cnt", LongType(), True),
    StructField("Down/Up Ratio", DoubleType(), True),
    StructField("Pkt Size Avg", DoubleType(), True),
    StructField("Fwd Seg Size Avg", DoubleType(), True),
    StructField("Bwd Seg Size Avg", DoubleType(), True),
    StructField("Fwd Byts/b Avg", DoubleType(), True),
    StructField("Fwd Pkts/b Avg", DoubleType(), True),
    StructField("Fwd Blk Rate Avg", DoubleType(), True),
    StructField("Bwd Byts/b Avg", DoubleType(), True),
    StructField("Bwd Pkts/b Avg", DoubleType(), True),
    StructField("Bwd Blk Rate Avg", DoubleType(), True),
    StructField("Subflow Fwd Pkts", LongType(), True),
    StructField("Subflow Fwd Byts", LongType(), True),
    StructField("Subflow Bwd Pkts", LongType(), True),
    StructField("Subflow Bwd Byts", LongType(), True),
    StructField("Init Fwd Win Byts", LongType(), True),
    StructField("Init Bwd Win Byts", LongType(), True),
    StructField("Fwd Act Data Pkts", LongType(), True),
    StructField("Fwd Seg Size Min", LongType(), True),
    StructField("Active Mean", DoubleType(), True),
    StructField("Active Std", DoubleType(), True),
    StructField("Active Max", DoubleType(), True),
    StructField("Active Min", DoubleType(), True),
    StructField("Idle Mean", DoubleType(), True),
    StructField("Idle Std", DoubleType(), True),
    StructField("Idle Max", DoubleType(), True),
    StructField("Idle Min", DoubleType(), True),
    StructField("Label", StringType(), True),
])

traffic_ip_schema = StructType([
    StructField("Flow ID", StringType(), True),
    StructField("Src IP", StringType(), True),
    StructField("Src Port", IntegerType(), True),
    StructField("Dst IP", StringType(), True),
] + traffic_schema.fields)

print("Original traffic schemas:", len(traffic_schema), "and", len(traffic_ip_schema), "columns")


Original traffic schemas: 80 and 84 columns


### IANA schema

We read IANA columns as text. A port can be a range such as `225-241`, so we cannot convert every value to an integer yet.


In [9]:
iana_schema = StructType([
    StructField("Service Name", StringType(), True),
    StructField("Port Number", StringType(), True),
    StructField("Transport Protocol", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Assignee", StringType(), True),
    StructField("Contact", StringType(), True),
    StructField("Registration Date", StringType(), True),
    StructField("Modification Date", StringType(), True),
    StructField("Reference", StringType(), True),
    StructField("Service Code", StringType(), True),
    StructField("Unauthorized Use Reported", StringType(), True),
    StructField("Assignment Notes", StringType(), True),
])

print("IANA schema:", len(iana_schema), "columns")

IANA schema: 12 columns


### Check each header against its schema

The column names and order must match. If they do not, stop and check the file. Step 3 will test whether the values can be read with these types.


In [10]:
traffic_schemas = {}
for file_name, header in traffic_headers.items():
    schema = traffic_ip_schema if "Flow ID" in header else traffic_schema
    assert header == schema.fieldNames(), f"Check the columns in {file_name}"
    traffic_schemas[file_name] = schema
    print(f"Header matches: {file_name} ({len(schema)} columns)")

assert iana_header == iana_schema.fieldNames(), "Check the IANA columns"
print("IANA header matches.")


Header matches: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv (84 columns)
Header matches: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
IANA header matches.


Step 2 is complete. Next, we test the schemas with the samples.


## Step 3 - Test CSV to Parquet

We use the 5,000 traffic rows from Step 1 and the whole IANA file to test reading CSV, writing Parquet, and reading it back. This checks the code before we process all the traffic files.


### 3A - Save the samples and start Spark

Save one small CSV per traffic file in `data/samples/`. Each keeps its own header. Start Spark with two local worker threads and four shuffle partitions.


In [11]:
sample_folder = data_folder / "samples"
sample_folder.mkdir(parents=True, exist_ok=True)
for path in traffic_files:
    with (sample_folder / path.name).open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=traffic_headers[path.name])
        writer.writeheader()
        writer.writerows(file_samples[path.name])

from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]")
         .appName("DPA-Volume")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Sample traffic records:", len(sample_rows))


Sample traffic records: 5000


### 3B - Read the CSVs

Read each file with its schema. `FAILFAST` stops if a value cannot be parsed. IANA needs `multiLine=True` because some quoted fields contain line breaks.

`unionByName` combines files using column names. `allowMissingColumns=True` fills the four extra columns with null where a file does not have them. These are not recovered IP addresses.

Use lowercase column names with underscores to make them easier to use.


In [12]:
sample_parts = []
for path in traffic_files:
    part = (spark.read.schema(traffic_schemas[path.name])
             .option("header", True).option("enforceSchema", False)
             .option("mode", "FAILFAST").option("escape", '"')
             .csv(str(sample_folder / path.name)))
    sample_parts.append(part)

sample_traffic = sample_parts[0]
for part in sample_parts[1:]:
    sample_traffic = sample_traffic.unionByName(part, allowMissingColumns=True)

iana_source = (spark.read.schema(iana_schema)
           .option("header", True).option("enforceSchema", False)
           .option("mode", "FAILFAST").option("escape", '"')
           .option("multiLine", True).csv(str(iana_file)))

sample_traffic = sample_traffic.toDF(*[name.lower().replace(" ", "_").replace("/", "_") for name in sample_traffic.columns])
iana_source = iana_source.toDF(*[name.lower().replace(" ", "_") for name in iana_source.columns])

sample_traffic.select("dst_port", "protocol", "flow_duration", "label").printSchema()
sample_traffic.select("dst_port", "protocol", "flow_duration", "label").show(5)


root
 |-- dst_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- label: string (nullable = true)



+--------+--------+-------------+------+
|dst_port|protocol|flow_duration| label|
+--------+--------+-------------+------+
|     443|       6|       141385|Benign|
|   49684|       6|          281|Benign|
|     443|       6|       279824|Benign|
|     443|       6|          132|Benign|
|     443|       6|       274016|Benign|
+--------+--------+-------------+------+
only showing top 5 rows


### 3C - Save and check Parquet

Parquet keeps the column types. Save the test tables in `data/parquet/sample/`, then read them back. Check the row counts, types, and contents. `exceptAll` compares rows while keeping duplicate counts.


In [13]:
sample_tables = parquet_folder / "sample"
sample_traffic.write.mode("overwrite").parquet(str(sample_tables / "traffic"))
iana_source.write.mode("overwrite").parquet(str(sample_tables / "iana"))

traffic_parquet = spark.read.parquet(str(sample_tables / "traffic"))
iana_parquet = spark.read.parquet(str(sample_tables / "iana"))

for name, original, restored, expected in [
    ("Traffic", sample_traffic, traffic_parquet, len(sample_rows)),
    ("IANA", iana_source, iana_parquet, len(iana_records)),
]:
    actual = restored.count()
    assert actual == expected, f"{name}: row count changed"
    assert original.dtypes == restored.dtypes, f"{name}: column types changed"
    assert original.exceptAll(restored).count() == 0, f"{name}: row contents changed"
    print(f"{name}: {actual} rows; types and contents preserved.")

print("Step 3 passed. Parquet saved in:", sample_tables)

Traffic: 5000 rows; types and contents preserved.


IANA: 14536 rows; types and contents preserved.
Step 3 passed. Parquet saved in: /home/jovyan/work/data/parquet/sample


Step 3 is complete. The sample tables passed the checks. Step 4 uses all original traffic rows, not these samples.


## Step 4 - Read and clean all the data

We read all ten original traffic files and the whole IANA file. We save typed Parquet before cleaning, then work from those tables. We leave the source CSVs unchanged.

Before each cleaning step, we explain why we need it. We record how many rows were removed or changed.

The [paper's abstract](https://www.nu.edu.eg/publications/apache-spark-powered-enhancing-network-intrusion-detection-system-using-random-forest) also mentions missing values, duplicates, and unused columns. We could not access the full methods. The rules below are our choices for our research questions.


### 4A - Read all files and save raw Parquet

**Why:** The sample only tested the code. Now we need all the data, using the correct schema for each file.

`PERMISSIVE` puts badly formatted records in `_corrupt_record`. We save them in raw Parquet so we can count them before removal. We disable CSV column pruning so Spark checks every field. `source_file` keeps the original filename.

The missing IP columns in nine files are expected. We will not remove rows because of them.


In [ ]:
from pyspark.sql import functions as F
from pyspark import StorageLevel

spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "16")
traffic_parts = []
for path in traffic_files:
    schema = StructType(traffic_schemas[path.name].fields + [StructField("_corrupt_record", StringType(), True)])
    part = (spark.read.schema(schema)
             .option("header", True).option("enforceSchema", False)
             .option("mode", "PERMISSIVE").option("escape", '"')
             .option("columnNameOfCorruptRecord", "_corrupt_record")
             .csv(str(path)).withColumn("source_file", F.lit(path.name)))
    traffic_parts.append(part)
traffic_source = traffic_parts[0]
for part in traffic_parts[1:]:
    traffic_source = traffic_source.unionByName(part, allowMissingColumns=True)
traffic_source = traffic_source.toDF(*[column.lower().replace(" ", "_").replace("/", "_") for column in traffic_source.columns])

raw_folder = parquet_folder / "raw"
traffic_source.write.mode("overwrite").parquet(str(raw_folder / "traffic"))
iana_source.write.mode("overwrite").parquet(str(raw_folder / "iana"))
raw_traffic = spark.read.parquet(str(raw_folder / "traffic"))
raw_iana = spark.read.parquet(str(raw_folder / "iana"))
print("Original files processed:", len(traffic_files))
raw_traffic.groupBy("source_file").count().orderBy("source_file").show(truncate=False)


### Keep a record of each step

For each rule, save the input, removed, changed, and remaining row counts, with a reason. Each removal count starts from the previous step, so a row removed earlier is not counted again.

Changing values or types counts as a modification. Selecting fewer columns does not. Steps that only change columns keep the same row count, so we do not need to count all the rows again.


In [ ]:
audit_rows = []

def record_step(dataset, step, before, after, reason, modified=0):
    audit_rows.append((dataset, step, before, before - after, modified, after, reason))
    print(f"{dataset} | {step}: {before:,} -> {after:,}; removed {before - after:,}; modified {modified:,}")
    return after

traffic = raw_traffic
iana = raw_iana
traffic_count = traffic.count()
iana_count = iana.count()
print(f"Input rows: traffic={traffic_count:,}, IANA={iana_count:,}")


### 4B - Remove badly formatted records

**Why:** Rows with the wrong number of fields or invalid numeric text do not match our schema. A repeated header inside a file is one example. We remove these rows from the working table. Their original text stays in raw Parquet.


In [ ]:
traffic = traffic.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
traffic_count = record_step("traffic", "Malformed records", traffic_count, traffic.count(),
                            "Remove CSV structure or numeric parsing failures; raw evidence retained")


### 4C - Remove duplicate records

**Why:** Repeated rows would be counted more than once in the analysis. We compare all source columns before removing any columns. For traffic, `source_file` means we check duplicates within each file.

These are matching CSV records. Nine files have no host IPs, so we cannot prove they are the same network event. For IANA, we also compare the complete rows first.


In [ ]:
traffic = traffic.dropDuplicates().persist(StorageLevel.DISK_ONLY)
traffic_count = record_step("traffic", "Duplicate records", traffic_count, traffic.count(),
                            "Keep one matching typed source record per file, comparing all source columns")
iana = iana.dropDuplicates()
iana_count = record_step("iana", "Duplicate records", iana_count, iana.count(),
                         "Keep one copy of each complete registry record")


### 4D - Keep the columns we need

**Why:** We need labels, dates, durations, packets, bytes, and rates to compare traffic. Destination port and protocol are needed for the service lookup. We also keep the filename.

For IANA, we need service name, port number, and transport protocol. We drop the other columns from the working tables. Raw Parquet still keeps every source column.

We save reused traffic tables on disk with `persist` to avoid repeating the duplicate check and to limit memory use. We release them with `unpersist` when finished.


In [ ]:
traffic_columns = ["dst_port", "protocol", "timestamp", "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",
                    "totlen_fwd_pkts", "totlen_bwd_pkts", "flow_byts_s", "flow_pkts_s", "label"]
unique_traffic = traffic
selected_traffic = unique_traffic.select(*traffic_columns, "source_file").persist(StorageLevel.DISK_ONLY)
traffic = selected_traffic
traffic_count = record_step("traffic", "Select analysis columns", traffic_count, traffic.count(),
                            "Keep 11 research fields and source_file; raw table retains all columns")
unique_traffic.unpersist()
iana = iana.select("service_name", "port_number", "transport_protocol")
iana_count = record_step("iana", "Select lookup columns", iana_count, iana_count,
                         "Keep service name, port number, and transport protocol")


### 4E - Fix spaces and empty text

**Why:** Extra spaces can split matching values into different groups. We trim labels, dates, and IANA fields. We lowercase IANA protocols and turn empty text into null. We keep the original attack-label spelling.


In [ ]:
changed_traffic_rows = traffic.filter(
    (F.col("label") != F.trim("label")) | (F.trim("label") == "") |
    (F.col("timestamp") != F.trim("timestamp")) | (F.trim("timestamp") == "")).count()
for column in ["label", "timestamp"]:
    traffic = traffic.withColumn(column, F.when(F.trim(column) != "", F.trim(column)))
traffic_count = record_step("traffic", "Normalize text", traffic_count, traffic_count,
                            "Trim labels/timestamps and turn blank text into null", changed_traffic_rows)

changed_iana_rows = iana.filter(
    (F.col("service_name") != F.trim("service_name")) | (F.trim("service_name") == "") |
    (F.col("port_number") != F.trim("port_number")) | (F.trim("port_number") == "") |
    (F.col("transport_protocol") != F.lower(F.trim("transport_protocol"))) |
    (F.trim("transport_protocol") == "")).count()
for column in iana.columns:
    iana = iana.withColumn(column, F.when(F.trim(column) != "", F.trim(column)))
iana = iana.withColumn("transport_protocol", F.lower("transport_protocol"))
iana_count = record_step("iana", "Normalize text", iana_count, iana_count,
                         "Trim lookup fields, lowercase protocols, and turn blank text into null", changed_iana_rows)


### 4F - Check and remove missing values

**Why:** Our comparisons need all 11 selected traffic fields. We print the missing counts, then drop rows with null or `NaN` in these fields. `NaN` means a number is undefined. Filling in labels, dates, or traffic rates would add guesses to our results.

The IANA lookup needs a service name, port, and protocol. We exclude rows missing these fields from the lookup, even though they may be valid registry entries. Missing IP columns are not part of this check.


In [ ]:
numeric_columns = [column for column in traffic_columns if column not in ["label", "timestamp"]]
missing_checks = []
for column in traffic_columns:
    missing = F.col(column).isNull()
    if column in numeric_columns:
        missing = missing | F.isnan(column)
    missing_checks.append(F.sum(F.when(missing, 1).otherwise(0)).alias(column))
missing_counts = traffic.select(*missing_checks).first().asDict()
print("Traffic missing values before this rule:", {column: n for column, n in missing_counts.items() if n})
print("IANA missing values before this rule:", iana.select(*[
    F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column) for column in iana.columns]).first().asDict())
traffic = traffic.na.drop(subset=traffic_columns)
traffic_count = record_step("traffic", "Missing required values", traffic_count, traffic.count(),
                            "Drop null/NaN in required research fields; do not impute traffic or labels")
iana = iana.na.drop(subset=["service_name", "port_number", "transport_protocol"])
iana_count = record_step("iana", "Missing lookup values", iana_count, iana.count(),
                         "A named-service lookup requires service, port, and transport protocol")


### 4G - Remove infinite values

**Why:** An infinite rate cannot be used in our comparisons. We remove rows with infinity in any selected numeric field. Missing values were removed first, so a row with both problems is counted only once.


In [ ]:
valid_numbers = F.lit(True)
for column in numeric_columns:
    valid_numbers = valid_numbers & (F.abs(F.col(column).cast("double")) != F.lit(float("inf")))
traffic = traffic.filter(valid_numbers)
traffic_count = record_step("traffic", "Infinite measurements", traffic_count, traffic.count(),
                            "Retained numeric measurements must be finite")


### 4H - Check impossible values

**Why:** Durations, packet counts, bytes, and rates cannot be negative. Ports must be from 0 to 65535, and protocol codes from 0 to 255. Positive bytes with zero packets in the same direction are also inconsistent.

We keep zero ports, zero durations, and protocols other than TCP/UDP. Large finite values can be part of an attack, so we do not remove them just because they are high. The unused TCP window columns can contain special negative values such as `-1`.


In [ ]:
valid_values = F.col("dst_port").between(0, 65535) & F.col("protocol").between(0, 255)
for column in ["flow_duration", "tot_fwd_pkts", "tot_bwd_pkts", "totlen_fwd_pkts", "totlen_bwd_pkts",
          "flow_byts_s", "flow_pkts_s"]:
    valid_values = valid_values & (F.col(column) >= 0)
valid_values = valid_values & ((F.col("tot_fwd_pkts") > 0) | (F.col("totlen_fwd_pkts") == 0))
valid_values = valid_values & ((F.col("tot_bwd_pkts") > 0) | (F.col("totlen_bwd_pkts") == 0))
traffic = traffic.filter(valid_values)
traffic_count = record_step("traffic", "Impossible measurements or codes", traffic_count, traffic.count(),
                            "Reject negative measurements, invalid port/protocol ranges, and bytes without packets")


### 4I - Convert and check dates

**Why:** We need timestamp values for time comparisons. We read day/month/year, including fractional seconds. We remove invalid dates and dates outside 14 February to 2 March 2018.

We check the whole collection period because traffic can cross midnight. The source gives no timezone offset, so we keep its clock times. Every retained text-to-timestamp conversion counts as a changed row.


In [ ]:
traffic = traffic.withColumn("timestamp", F.coalesce(
    F.try_to_timestamp("timestamp", F.lit("dd/MM/yyyy HH:mm:ss")),
    F.try_to_timestamp("timestamp", F.lit("dd/MM/yyyy HH:mm:ss.SSSSSS"))))
traffic = traffic.filter(
    F.col("timestamp").isNotNull() &
    (F.col("timestamp") >= F.lit("2018-02-14").cast("timestamp")) &
    (F.col("timestamp") < F.lit("2018-03-03").cast("timestamp")))
date_count = traffic.count()
traffic_count = record_step("traffic", "Parse and validate timestamps", traffic_count, date_count,
                            "Convert valid timestamps; reject invalid dates outside the collection period", date_count)


### 4J - Prepare individual IANA ports

**Why:** Each traffic row has one destination port. For a simple join, we keep individual TCP/UDP ports, convert them to integers, and check the range 0 to 65535.

Port ranges and other protocols are valid IANA entries, but this lookup does not use them. Later, an unmatched flow will not necessarily mean an unregistered port.


In [ ]:
iana = iana.filter(
    F.col("port_number").rlike(r"^[0-9]{1,5}$") & F.col("transport_protocol").isin("tcp", "udp"))
iana = iana.withColumn("port_number", F.col("port_number").cast("int"))
iana = iana.filter(F.col("port_number").between(0, 65535))
port_count = iana.count()
iana_count = record_step("iana", "Individual TCP/UDP ports", iana_count, port_count,
                         "Lookup scope: individual TCP/UDP registrations; convert retained ports to integers", port_count)


### 4K - Combine service names sharing a port

**Why:** Several service names can share a port and protocol. Joining all of them separately would multiply traffic rows. We keep the names together in one sorted string per port and protocol. These aliases are valid names, not duplicate errors.


In [ ]:
services_by_port = iana.groupBy("port_number", "transport_protocol").agg(
    F.sort_array(F.collect_set("service_name")).alias("service_names"))
changed_services = services_by_port.filter(F.size("service_names") > 1).count()
iana = services_by_port.select("port_number", "transport_protocol",
                                 F.concat_ws(", ", "service_names").alias("service_name"))
iana_count = record_step("iana", "Combine service aliases", iana_count, iana.count(),
                         "Aggregate all distinct service names into one row per port/protocol; prevent join multiplication", changed_services)


### 4L - Save the results

Save the cleaned tables as Parquet. Read them back and check counts and types. Save the cleaning counts and reasons in one CSV. Rerunning replaces these generated files.


In [ ]:
clean_folder = parquet_folder / "cleaned"
traffic.write.mode("overwrite").parquet(str(clean_folder / "traffic"))
iana.write.mode("overwrite").parquet(str(clean_folder / "iana"))
clean_traffic = spark.read.parquet(str(clean_folder / "traffic"))
clean_iana = spark.read.parquet(str(clean_folder / "iana"))
assert clean_traffic.count() == traffic_count
assert clean_iana.count() == iana_count
assert clean_traffic.dtypes == traffic.dtypes
assert clean_iana.dtypes == iana.dtypes
selected_traffic.unpersist()

audit_columns = ["dataset", "step", "input_rows", "removed_rows", "modified_rows", "output_rows", "reason"]
cleaning_audit = spark.createDataFrame(audit_rows, audit_columns)
cleaning_audit.show(truncate=False)
audit_folder = project_folder / "reports" / "cleaning" / "original"
audit_folder.mkdir(parents=True, exist_ok=True)
with (audit_folder / "cleaning_audit.csv").open("w", encoding="utf-8", newline="") as output:
    writer = csv.writer(output)
    writer.writerow(audit_columns)
    writer.writerows(audit_rows)
print("Cleaned Parquet:", clean_folder)
print("Cleaning audit:", audit_folder / "cleaning_audit.csv")


After Step 4 runs, the cleaned traffic and IANA tables are ready for the join below. The audit gives the actual cleaning counts.


## Step 5 - Add IANA service names

We use the cleaned Parquet tables from Step 4. We add a registered service name using destination port and protocol. These names come from our downloaded registry and may differ from those used in 2018.

A service name is a clue about the traffic, not proof of the application being used. Run this step after Step 4. If the cleaned tables already exist, we can reuse them without cleaning again, as long as `project_folder` is set and Spark is running.


### 5A - Prepare the join columns

**Why:** Traffic stores protocols as numbers, but IANA uses names. We map `6` to `tcp` and `17` to `udp`. Other protocols keep their numeric code and have no TCP/UDP name.

We rename IANA's `port_number` to `dst_port` so both tables use the same join column. We check that each port and protocol has only one lookup row. Step 4 already combined service aliases.


In [28]:
from pyspark.sql import functions as F

clean_folder = parquet_folder / "cleaned"
traffic = spark.read.parquet(str(clean_folder / "traffic"))
iana = spark.read.parquet(str(clean_folder / "iana"))

traffic = traffic.withColumn(
    "transport_protocol",
    F.when(F.col("protocol") == 6, "tcp").when(F.col("protocol") == 17, "udp"))
iana = iana.withColumnRenamed("port_number", "dst_port")
assert iana.groupBy("dst_port", "transport_protocol").count().filter(
    F.col("count") > 1).count() == 0, "IANA has repeated join keys. Check Step 4."

rows_before = traffic.count()
print(f"Traffic rows before the join: {rows_before:,}")


Traffic rows before the join: 15,727,874


### 5B - Join traffic with IANA

**Why:** A left join keeps every traffic row, even when no service name matches. We join on both port and protocol because TCP and UDP can use the same port for different registrations.

IANA is small, so `broadcast` copies the lookup to Spark workers and avoids shuffling the large traffic table. Unmatched service names stay null. This does not automatically mean an unregistered port: our lookup leaves out port ranges and other protocols.


In [29]:
traffic_with_services = traffic.join(
    F.broadcast(iana), ["dst_port", "transport_protocol"], "left")


### 5C - Save and check the result

Save the enriched traffic as Parquet, then read it back. Check the types and confirm that the row count is unchanged. Count matched and unmatched rows so we know how much traffic the lookup covers. These are flow counts, not counts of different services.


In [30]:
enriched_folder = parquet_folder / "enriched" / "traffic"
traffic_with_services.write.mode("overwrite").parquet(str(enriched_folder))
final_traffic = spark.read.parquet(str(enriched_folder))
assert final_traffic.dtypes == traffic_with_services.dtypes, "Column types changed after saving."

service_counts = final_traffic.groupBy(
    F.col("service_name").isNotNull().alias("iana_match")).count().collect()
rows_after = sum(row["count"] for row in service_counts)
matched_rows = sum(row["count"] for row in service_counts if row["iana_match"])
assert rows_after == rows_before, "The join changed the traffic row count."
print(f"Traffic rows after the join: {rows_after:,}")
print(f"Matched service: {matched_rows:,}")
print(f"No service match: {rows_after - matched_rows:,}")

final_traffic.select("dst_port", "protocol", "transport_protocol", "service_name", "label").show(10, truncate=False)
print("Enriched Parquet:", enriched_folder)


Traffic rows after the join: 15,727,874
Matched service: 12,805,625
No service match: 2,922,249


+--------+--------+------------------+-------------------+------+
|dst_port|protocol|transport_protocol|service_name       |label |
+--------+--------+------------------+-------------------+------+
|80      |6       |tcp               |http, www, www-http|Benign|
|443     |6       |tcp               |https              |Benign|
|443     |6       |tcp               |https              |Benign|
|50574   |6       |tcp               |NULL               |Benign|
|50623   |6       |tcp               |NULL               |Benign|
|443     |6       |tcp               |https              |Benign|
|443     |6       |tcp               |https              |Benign|
|443     |6       |tcp               |https              |Benign|
|443     |6       |tcp               |https              |Benign|
|53      |17      |udp               |domain             |Benign|
+--------+--------+------------------+-------------------+------+
only showing top 10 rows
Enriched Parquet: /home/jovyan/work/data/parquet/en

After these checks pass, Step 5 is complete. The saved `final_traffic` table is ready for the research analysis below.


## Step 6 - Research analysis

We answer our two questions using all the cleaned, enriched traffic. These are recorded flows, not individual packets or unique hosts. We use the source labels as given: `Benign` means normal traffic, and the other labels describe attacks.

The results describe this dataset after cleaning. They do not measure attack rates on a real network or prove that one value can identify every attack.


### 6A - Prepare the measurements

**Why:** Add forward and backward counts to get total packets and bytes per flow. The [CICFlowMeter feature list](https://raw.githubusercontent.com/ahlashkari/CICFlowMeter/master/ReadMe.txt) gives flow duration in microseconds, so we divide by 1,000 to show milliseconds. The supplied rate columns are bytes per second and packets per second.

We read the saved enriched table. Earlier cleaning does not need to run again if this table exists, `parquet_folder` is set, and Spark is running.

We keep only the ten columns used in the analysis.


In [3]:
from pyspark.sql import functions as F

traffic = spark.read.parquet(str(parquet_folder / "enriched" / "traffic")).select(
    "label", "dst_port", "protocol", "service_name",
    F.when(F.col("label") == "Benign", "Normal").otherwise("Attack").alias("traffic_type"),
    (F.col("flow_duration") / 1000).alias("duration_ms"),
    (F.col("tot_fwd_pkts") + F.col("tot_bwd_pkts")).alias("packets"),
    (F.col("totlen_fwd_pkts") + F.col("totlen_bwd_pkts")).alias("bytes"),
    "flow_byts_s", "flow_pkts_s")


### 6B - Summarize each label once

**Why:** Counts, measurements, and service-match counts use the same label groups. Calculate them together and cache the small summary. The following cells reuse it.

We show four decimal places for percentages so rare labels remain visible. A large flow count may reflect the capture setup; it does not show how dangerous an attack is.


In [4]:
measurements = [
    F.count("*").alias("flows"),
    F.round(F.avg("duration_ms"), 3).alias("avg_duration_ms"),
    F.round(F.percentile_approx("duration_ms", 0.5), 3).alias("median_duration_ms"),
    F.round(F.avg("packets"), 2).alias("avg_packets"),
    F.percentile_approx("packets", 0.5).alias("median_packets"),
    F.round(F.avg("bytes"), 2).alias("avg_bytes"),
    F.percentile_approx("bytes", 0.5).alias("median_bytes"),
    F.round(F.percentile_approx("flow_byts_s", 0.5), 2).alias("median_bytes_s"),
    F.round(F.percentile_approx("flow_pkts_s", 0.5), 2).alias("median_packets_s"),
]

label_summary = (traffic.groupBy("label")
    .agg(*measurements,
         F.sum(F.when(F.col("service_name").isNotNull(), 1).otherwise(0)).alias("matched_flows"))
    .orderBy(F.desc("flows"), "label").cache())
total_flows = label_summary.agg(F.sum("flows")).first()[0]
assert total_flows is not None and total_flows > 0, "The enriched traffic table is empty."
print(f"Flows used in the analysis: {total_flows:,}")

label_counts = label_summary.select("label", "flows").withColumn(
    "percent_of_all", F.round(F.col("flows") / total_flows * 100, 4))
label_counts.show(30, truncate=False)


Flows used in the analysis: 15,727,874


+------------------------+--------+--------------+
|label                   |flows   |percent_of_all|
+------------------------+--------+--------------+
|Benign                  |13374907|85.0395       |
|DDOS attack-HOIC        |668461  |4.2502        |
|DDoS attacks-LOIC-HTTP  |576191  |3.6635        |
|DoS attacks-Hulk        |434873  |2.765         |
|Bot                     |282310  |1.795         |
|Infilteration           |160604  |1.0211        |
|SSH-Bruteforce          |117322  |0.7459        |
|DoS attacks-GoldenEye   |41455   |0.2636        |
|FTP-BruteForce          |39346   |0.2502        |
|DoS attacks-SlowHTTPTest|19462   |0.1237        |
|DoS attacks-Slowloris   |10285   |0.0654        |
|DDOS attack-LOIC-UDP    |1730    |0.011         |
|Brute Force -Web        |611     |0.0039        |
|Brute Force -XSS        |230     |0.0015        |
|SQL Injection           |87      |6.0E-4        |
+------------------------+--------+--------------+



### 6C - Compare normal traffic and attacks

**Why:** Compare duration, packets, bytes, and rates to answer the first question. Averages show the overall level, while medians describe a typical flow with less influence from large values. `percentile_approx` calculates approximate medians in Spark.

First compare normal traffic with all attacks. Then use the same measurements for each label, because different attack types may have different patterns. We keep the source rate values instead of calculating new rates for zero-duration flows.

The Benign row already contains the Normal statistics, so we reuse it. We calculate the combined Attack statistics from the attack flows. This avoids processing normal flows a second time.


In [5]:
normal_stats = (label_summary.filter(F.col("label") == "Benign")
    .drop("label", "matched_flows").withColumn("traffic_type", F.lit("Normal")))
attack_stats = (traffic.filter(F.col("label") != "Benign")
    .groupBy("traffic_type").agg(*measurements))
normal_attack = attack_stats.unionByName(normal_stats).orderBy("traffic_type").cache()
normal_attack.show(truncate=False)

label_patterns = label_summary.drop("matched_flows")
label_patterns.show(30, truncate=False)


+------------+--------+---------------+------------------+-----------+--------------+---------+------------+--------------+----------------+
|traffic_type|flows   |avg_duration_ms|median_duration_ms|avg_packets|median_packets|avg_bytes|median_bytes|median_bytes_s|median_packets_s|
+------------+--------+---------------+------------------+-----------+--------------+---------+------------+--------------+----------------+
|Attack      |2352967 |4393.841       |15.269            |129.8      |2             |4612.06  |0.0         |0.0           |166.82          |
|Normal      |13374907|14019.113      |37.07             |13.31      |4             |6111.21  |179.0       |1161.89       |103.73          |
+------------+--------+---------------+------------------+-----------+--------------+---------+------------+--------------+----------------+



+------------------------+--------+---------------+------------------+-----------+--------------+----------+------------+--------------+----------------+
|label                   |flows   |avg_duration_ms|median_duration_ms|avg_packets|median_packets|avg_bytes |median_bytes|median_bytes_s|median_packets_s|
+------------------------+--------+---------------+------------------+-----------+--------------+----------+------------+--------------+----------------+
|Benign                  |13374907|14019.113      |37.07             |13.31      |4             |6111.21   |179.0       |1161.89       |103.73          |
|DDOS attack-HOIC        |668461  |9.842          |6.733             |3.22       |2             |302.32    |0.0         |0.0           |474.95          |
|DDoS attacks-LOIC-HTTP  |576191  |11889.809      |1750.079          |159.06     |7             |5439.62   |984.0       |520.12        |3.83            |
|DoS attacks-Hulk        |434873  |320.317        |15.378            |2.5   

### 6D - Check service matches for each label

**Why:** Some labels may have fewer IANA matches than others. We show this before interpreting service results. A missing match does not mean an unregistered service or an attack. Port ranges and protocols outside TCP/UDP are not covered by our lookup.

These counts come from `label_summary`, so this cell does not scan the traffic again.


In [6]:
service_coverage = (label_summary.select("label", "flows", "matched_flows")
    .withColumn("matched_percent", F.round(F.col("matched_flows") / F.col("flows") * 100, 2)))
service_coverage.show(30, truncate=False)


+------------------------+--------+-------------+---------------+
|label                   |flows   |matched_flows|matched_percent|
+------------------------+--------+-------------+---------------+
|Benign                  |13374907|10488198     |78.42          |
|DDOS attack-HOIC        |668461  |668461       |100.0          |
|DDoS attacks-LOIC-HTTP  |576191  |576191       |100.0          |
|DoS attacks-Hulk        |434873  |434873       |100.0          |
|Bot                     |282310  |277753       |98.39          |
|Infilteration           |160604  |129643       |80.72          |
|SSH-Bruteforce          |117322  |117322       |100.0          |
|DoS attacks-GoldenEye   |41455   |41455        |100.0          |
|FTP-BruteForce          |39346   |39346        |100.0          |
|DoS attacks-SlowHTTPTest|19462   |19462        |100.0          |
|DoS attacks-Slowloris   |10285   |10285        |100.0          |
|DDOS attack-LOIC-UDP    |1730    |1730         |100.0          |
|Brute For

### 6E - Find the top ports and services for each label

**Why:** This answers the service part of our second question. Count flows for each destination port and protocol, then keep the five most common combinations within each label, including normal traffic. Percentages use all flows in that label, including unmatched rows.

`row_number` ranks the ports separately for each label. We break ties by port and protocol so the order is consistent. Service aliases stay together and unmatched ports remain in the result. IANA names describe registered port associations, not verified applications.

We keep the top five ports before joining label totals and calculating percentages. That work now runs only on the shortlist.


In [7]:
from pyspark.sql.window import Window

service_counts = traffic.groupBy("label", "dst_port", "protocol", "service_name").agg(
    F.count("*").alias("flows"))
port_order = Window.partitionBy("label").orderBy(F.desc("flows"), "dst_port", "protocol")
top_ports = service_counts.withColumn("rank", F.row_number().over(port_order)).filter(F.col("rank") <= 5)

label_totals = label_counts.select("label", F.col("flows").alias("label_flows"))
top_services = (top_ports.join(F.broadcast(label_totals), "label", "left")
    .withColumn("percent_of_label", F.round(F.col("flows") / F.col("label_flows") * 100, 2))
    .withColumn("service_name", F.coalesce("service_name", F.lit("No IANA match")))
    .select("label", "rank", "dst_port", "protocol", "service_name", "flows", "percent_of_label")
    .orderBy("label", "rank").cache())
top_services.show(100, truncate=False)


+------------------------+----+--------+--------+-------------------+-------+----------------+
|label                   |rank|dst_port|protocol|service_name       |flows  |percent_of_label|
+------------------------+----+--------+--------+-------------------+-------+----------------+
|Benign                  |1   |53      |17      |domain             |4020801|30.06           |
|Benign                  |2   |443     |6       |https              |1995818|14.92           |
|Benign                  |3   |3389    |6       |ms-wbt-server      |1819697|13.61           |
|Benign                  |4   |80      |6       |http, www, www-http|1678562|12.55           |
|Benign                  |5   |445     |6       |microsoft-ds       |706291 |5.28            |
|Bot                     |1   |8080    |6       |http-alt           |277753 |98.39           |
|Bot                     |2   |0       |0       |No IANA match      |258    |0.09            |
|Bot                     |3   |50891   |6       |N

### 6F - Save the small result tables

Save one CSV per summary in `reports/analysis/` for the report and charts. Only these small tables are collected into Python. All grouping and calculations use Spark. We release the cached results after saving them.

Only three small tables are cached: the label summary, the normal/attack comparison, and the top services.


In [8]:
import csv

results_folder = project_folder / "reports" / "analysis"
results_folder.mkdir(parents=True, exist_ok=True)
results = {"label_counts": label_counts, "normal_attack": normal_attack,
           "label_patterns": label_patterns, "service_coverage": service_coverage,
           "top_services": top_services}
for name, table in results.items():
    with (results_folder / f"{name}.csv").open("w", encoding="utf-8", newline="") as output:
        writer = csv.writer(output)
        writer.writerow(table.columns)
        writer.writerows(tuple(row) for row in table.collect())
for table in [label_summary, normal_attack, top_services]:
    table.unpersist()
print("Analysis results:", results_folder)


Analysis results: /home/jovyan/work/reports/analysis


### Our answers

**Question 1 - What traffic patterns distinguish normal activity from attacks?**

In this cleaned dataset:

| Traffic | Flows | Median duration (ms) | Median packets | Average packets |
| --- | ---: | ---: | ---: | ---: |
| Normal | 13,374,907 | 37.070 | 4 | 13.31 |
| Attack | 2,352,967 | 15.269 | 2 | 129.80 |

Attack flows have a shorter median duration and fewer packets per typical flow here, but a much higher average packet count. This shows why averages alone can be misleading.

The attack types also differ: HOIC's median duration is 6.733 ms, while Slowloris is about 105.08 seconds. The combined attack summary hides these differences.

**Question 2 - Which services and traffic characteristics are associated with each attack category?**

The per-label measurements above compare their durations, packets, bytes, and rates. Some clear port associations are:

| Label | Most common destination | Share of that label |
| --- | --- | ---: |
| DDOS attack-HOIC | 80/TCP | 100.00% |
| FTP-BruteForce | 21/TCP | 100.00% |
| SSH-Bruteforce | 22/TCP | 99.99% |
| Bot | 8080/TCP | 98.39% |

Port 80/TCP also accounts for 12.55% of normal flows. A port name alone does not separate normal traffic from attacks.

**Points to keep in mind:**

- The data is uneven: 85.04% of flows are normal, while SQL Injection has only 87 flows. Rare-label results have much less data behind them.
- Several attack labels have zero median recorded bytes despite having packets. These are the stored feature values; zero bytes does not mean no packets were sent.
- All SlowHTTPTest-labelled rows use port 21/TCP, which IANA names FTP. This is an observed label/port mismatch to check before making claims about the application. We report the supplied label and measured port.
- These summaries describe associations in this dataset. They are not a tested detection rule, and the IANA names do not prove which application was used.

Next: make charts from these summaries, then compare CSV and Parquet performance.
